# Python程序设计（项目实战） · 工单P5《产线数据仓库与ORM》 — 教师参考版
第7周 | 参考实现可直接运行验证 | 答案请勿下发学生

In [ ]:
# --- 环境与数据准备(勿改, 运行即可) ---
import os, sys, subprocess
os.chdir(os.path.abspath("../"))          # 切到课程包根目录(data/所在处)
print("工作目录:", os.getcwd())


## 参考实现 `p55`（含依赖导入）

In [ ]:
import sys
import numpy as np  # 通用科学计算


def p55():
    from sqlalchemy import create_engine, select, func
    from sqlalchemy.orm import DeclarativeBase, Session, relationship
    from sqlalchemy import Column, DateTime, Float, ForeignKey, Integer, String
    import datetime
    import random

    class Base(DeclarativeBase):
        pass

    class Dev(Base):
        __tablename__ = "devices"
        id = Column(Integer, primary_key=True)
        code = Column(String, unique=True)

    class Rec(Base):
        __tablename__ = "records"
        id = Column(Integer, primary_key=True)
        device_id = Column(Integer, ForeignKey("devices.id"))
        ts = Column(DateTime, default=datetime.datetime.utcnow)
        vib = Column(Float)

    e = create_engine("sqlite:///ref_warehouse.db")
    Base.metadata.drop_all(e)
    Base.metadata.create_all(e)
    random.seed(1)
    with Session(e) as s:
        s.add_all([Dev(code=f"DEV-{i:02d}") for i in range(1, 4)])
        s.add_all([Rec(device_id=random.randint(1, 3),
                       ts=datetime.datetime(2026, 4, 1) +
                       datetime.timedelta(minutes=i),
                       vib=random.uniform(1, 6)) for i in range(500)])
        s.commit()
        print("按设备平均振动:",
              s.execute(select(Dev.code, func.avg(Rec.vib))
                        .join(Rec, Rec.device_id == Dev.id)
                        .group_by(Dev.code)).all())
    with e.connect() as c:
        import time
        t0 = time.time()
        c.execute(select(Rec).where(Rec.device_id == 1).limit(200)).fetchall()
        print("查询耗时 %.2fms (加索引后对比见工单)" % ((time.time() - t0) * 1000))


## 执行与验证

In [ ]:
p55()